# 07 — Leave-One-Language-Out Parity Calibration (Table 8)

Notebook 05 ruled out per-language correction. This notebook asks a harder
question: can the parity features themselves — TP, IP and their native-minus-
romanised deltas — predict what the native score *would have been*, for a
language the model has never seen?

A gradient-boosted regressor is fitted on four languages to predict native COMET
from romanised COMET plus parity features, then applied to the held-out fifth.
Recovery is measured on the scale between the raw romanised correlation and the
native ceiling:

    recovered = (ρ_calibrated − ρ_raw) / (ρ_native − ρ_raw)

**The answer is: partially, and unreliably.** Mean recovery is 17.1%, and Marathi
is *negative* — calibration actively degrades it. Parity features carry real
signal about script burden, but not enough to substitute for scoring in the
native script.

**Base:** 6,995. GBM: 300 trees, depth 3, learning rate 0.05, `random_state=0`.

**Input:** `../data/indic/indic_parity_xlmr.xlsx`
**Output:** `../results/tables/table8_lolo_calibration.csv`

> RNG-dependent; registered with status `script` in `paper_numbers.yaml`.

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)


# ── GBM hyper-parameters ─────────────────────────────────────────────────────
GBM_PARAMS = dict(n_estimators=300, max_depth=3, learning_rate=0.05,
                  random_state=SEED_GBM)

## Step 1 — Load and Assemble the Working Set

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor

A = pd.concat([work[l].assign(lang=l) for l in LANG_ORDER], ignore_index=True)
A["Cn"]  = A[COL_COMET_NAT]
A["Cr"]  = A[COL_COMET_ROM]
A["TPn"] = A[COL_TP_NAT]
A["TPr"] = A[COL_TP_ROM]
A["IPn"] = A[COL_IP_NAT]
A["IPr"] = A[COL_IP_ROM]
A["dTP"] = A["TPn"] - A["TPr"]      # native minus romanised parity delta
A["dIP"] = A["IPn"] - A["IPr"]
print(f"Working set assembled: N = {len(A):,}")

## Step 2 — Feature Sets

The feature set is romanised COMET, both parity deltas, and all four raw parity
values — seven columns in total, as used by `scripts/reproduce_all.py`.

The native TP and IP columns matter: they carry the signal about how much burden
the native script was already under, which is what lets the model estimate the
scoring residual for a language it has not seen.

In [ ]:
# Seven parity features. Deltas are native minus romanised.
FEATURES = ["Cr", "dTP", "dIP", "TPr", "IPr", "TPn", "IPn"]
print("Parity feature set:", FEATURES)

## Step 3 — The Leave-One-Language-Out Loop

In [ ]:
def lolo(features, label):
    rows = []
    print(f"{label}")
    print(f"{'Held-out':>9}  {'\u03c1_raw':>8}  {'\u03c1_cal':>8}  {'\u03c1_nat':>8}  {'recovered':>10}")
    print("-" * 50)
    for lang in LANG_ORDER:
        tr, te = A[A.lang != lang], A[A.lang == lang]
        model = GradientBoostingRegressor(**GBM_PARAMS).fit(tr[features], tr["Cn"])
        pred = model.predict(te[features])
        r_raw = stats.spearmanr(te["Cr"], te["H"])[0]
        r_cal = stats.spearmanr(pred, te["H"])[0]
        r_nat = stats.spearmanr(te["Cn"], te["H"])[0]
        rec = (r_cal - r_raw) / (r_nat - r_raw) * 100
        rows.append(dict(lang=lang, rho_raw=r_raw, rho_cal=r_cal,
                         rho_nat=r_nat, recovered_pct=rec))
        print(f"{lang:>9}  {r_raw:>8.3f}  {r_cal:>8.3f}  {r_nat:>8.3f}  {rec:>9.1f}%")
    frame = pd.DataFrame(rows).set_index("lang")
    mean_rec = frame["recovered_pct"].mean()
    print("-" * 50)
    print(f"{'MEAN':>9}  {'':>8}  {'':>8}  {'':>8}  {mean_rec:>9.1f}%")
    return frame, mean_rec


table9, mean_recovered = lolo(FEATURES, "Table 8 — LOLO parity calibration (seven parity features)")

## Step 4 — Cross-Verify

In [ ]:
assert abs(mean_recovered - 17.1) < 0.1,     f"mean recovery = {mean_recovered:.1f}%, expected 17.1%"
assert table9.loc["MAR", "recovered_pct"] < 0, "MAR recovery should be negative"
assert (table9["rho_nat"] > table9["rho_raw"]).all()

print(f"\u2713 Mean recovery = {mean_recovered:.1f}% (reported: 17.1%)")
print(f"\u2713 MAR recovery = {table9.loc['MAR', 'recovered_pct']:+.1f}% — calibration "
      f"actively degrades Marathi")
print(f"\u2713 Best case is GUJ at {table9['recovered_pct'].max():.1f}%; even that leaves "
      f"most of the native-script advantage unrecovered")

## Step 6 — Save

In [ ]:
combined = table9.copy()
combined.loc["MEAN"] = [np.nan, np.nan, np.nan, mean_recovered]

path = TABLES_DIR / "table8_lolo_calibration.csv"
combined.to_csv(path)
print(combined.round(3).to_string())
print(f"\nSaved \u2192 {path}")

## Step 7 — Output Manifest

In [ ]:
print("=== Notebook 07 — output manifest ===")
print("  table8_lolo_calibration.csv")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1